# BTCUSDC Price-Level Survival Study

This notebook studies **price-level survival**, not changing layer labels.

Core question:

- If a price is visible at level `k` at time `t0`, when do we next observe that exact price disappear from the tracked book?
- Before that happens, how much of the initial displayed quantity is explained by aggressive trades at that same price, and how much is left as non-trade removal?


## Data Created

This notebook creates five main tables.

- `dataset`: lazy day descriptor from `load_day(...)`. It reads metadata and paths, but it does not replay the book yet.
- `trades`: enriched trade prints from `get_or_build_trades_table(...)`. Each row is a trade with aggressor side inferred from the feed.
- `book_levels`: replayed book states from `get_or_build_book_levels_table(...)`. Each row is one replayed book update with `bid1`, `ask1`, ..., up to the tracked depth.
- `price_levels`: one row per initial price-level observation. Each row starts from one snapshot and one initial level, then tracks that exact price forward until it disappears, falls below the tracked depth window, or the sample ends. There is no fixed forward horizon.
- `price_level_summary`: grouped summary of `price_levels` by aggressor side and initial level. It tells you whether the exact price observed at `t0` survived, disappeared, or was censored, and how long disappearance took when it happened.
- `queue_match_summary`: grouped summary of `price_levels` by aggressor side and initial level. It tells you whether the initial displayed queue at that price was plausibly worked through by aggressive flow, and how long that took under two definitions: trades only, or trades plus implied non-trade removal.


## Methodology

For every replayed book snapshot:

1. Choose the initial levels to study, for example levels `1..5`.
2. For each chosen level, record the exact price and displayed quantity at `t0`.
3. Track that **same price** forward through later replayed book states. The layer index is allowed to change as long as the price remains visible inside the tracked depth window.
4. Stop when one of two market outcomes occurs, or one censoring condition occurs:
   - `disappeared`: the price is absent, and from the tracked depth we can conclude it is gone from the visible book.
   - `survived_to_end`: the price is still visible when the sample ends.
   - `fell_below_window`: not a market outcome; we lost visibility because the price moved deeper than the tracked `top_n` levels. Treat this as right-censoring by depth.
5. Sum aggressive opposite-side trade volume at that same price until the stopping point.

Important interpretation:

- `observed_trade_qty_at_price` is the aggressive trade volume seen at the exact price before termination.
- `trade_explained_ratio = min(observed_trade_qty_at_price, initial_qty) / initial_qty`; it is the fraction of the starting queue explained by trades alone.
- `residual_nontrade_ratio = 1 - trade_explained_ratio` for `disappeared` rows only; it is the leftover fraction that is not explained by trades and is treated as implied non-trade removal pressure.
- `censored = True` means we do not observe the disappearance event; `censor_reason` tells whether that happened because the sample ended or because the price moved below the tracked depth window.

This residual should be read as **non-trade removal / cancellation / replacement pressure**, not literal cancel messages. If new liquidity was added at the same price before disappearance, the trade share is only a lower-bound style explanation of the initial displayed queue.

The two summary tables below condense the row-level results:

- `price_level_summary` asks whether the exact price seen at `t0` survived, disappeared, or was censored.
- `queue_match_summary` asks whether the initial displayed queue at that price was plausibly worked through by aggressive flow.
- `trade_match_share` is the fraction of rows where trades alone were enough to match the initial displayed quantity.
- `queue_match_share` is the fraction of rows where trades plus implied non-trade removals were enough to match the initial displayed quantity.
- the gap `queue_match_share - trade_match_share` is a proxy for how often non-trade queue turnover mattered; it is not a literal cancellation rate.

Queue-time proxies built on top of the same observations:

- `time_to_trade_matched_ms`: first time aggressive trades at the exact price reach the initial displayed quantity. This is the conservative trade-only wait estimate.
- `time_to_queue_matched_ms`: first time aggressive trades plus implied non-trade removal reach the initial displayed quantity. This is the closer FIFO-style queue-consumption estimate.


In [1]:
from pathlib import Path
import sys


def find_backtester_root() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "stats").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the exchange-data-backtester project root")


PROJECT_ROOT = find_backtester_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from IPython.display import display

from stats.analysis import (
    compute_price_level_survival,
    summarize_price_level_queue_matches,
    summarize_price_level_survival,
)
from stats.io import load_day
from stats.tables import get_or_build_book_levels_table, get_or_build_trades_table


In [2]:
DATA_ROOT = PROJECT_ROOT.parent / "exchange-data-recorder" / "data"
SYMBOL = "BTCUSDC"
DAY = "20260223"
DAY_DIR = DATA_ROOT / "binance" / SYMBOL / DAY

# `tracked_top_n` controls how deep we replay and observe the book.
# It is the maximum visible depth used for survival tracking and depth-window censoring.
tracked_top_n = 20
# `max_initial_level` controls which starting levels we study from each snapshot.
# `max_initial_level = 5` means we build initial observations for levels 1 through 5.
max_initial_level = 10

dataset = load_day(DAY_DIR)
trades = get_or_build_trades_table(dataset)
book_levels = get_or_build_book_levels_table(
    dataset,
    top_n=tracked_top_n,
    on_gap="skip-segment",
    show_progress=True,
)

print("Day dir:", dataset.day_dir)
print("Trades rows:", len(trades))
print("Book rows:", len(book_levels))
display(book_levels.head(3))


Day dir: /Users/hoangdeveloper/PycharmProjects/exchange-data-recorder/data/binance/BTCUSDC/20260223
Trades rows: 868008
Book rows: 722893


,event_type,recv_seq,recv_time_ms,event_time_ms,epoch_id,segment_index,segment_tag,bid1_price,bid1_qty,ask1_price,...,ask18_qty,bid19_price,bid19_qty,ask19_price,ask19_qty,bid20_price,bid20_qty,ask20_price,ask20_qty,ts
0,book,8,1771808404735,1771808404732,0,0,initial,66704.43,0.24012,66704.44,...,0.00017,66686.28,0.09824,66720.42,0.53903,66685.33,0.0624,66720.60,0.00009,2026-02-23 01:00:04.735000+00:00
1,book,11,1771808404835,1771808404832,0,0,initial,66704.43,0.24012,66704.44,...,0.00009,66688.08,0.53903,66720.63,0.02000,66686.43,0.0800,66720.83,0.08960,2026-02-23 01:00:04.835000+00:00
2,book,12,1771808404935,1771808404932,0,0,initial,66704.43,0.23310,66704.44,...,0.00009,66688.08,0.53903,66720.63,0.02000,66686.43,0.0800,66720.83,0.08960,2026-02-23 01:00:04.935000+00:00


In [3]:
price_levels = compute_price_level_survival(
    book_levels,
    trades,
    max_initial_level=max_initial_level,
    tracked_top_n=tracked_top_n,
)
price_level_summary = summarize_price_level_survival(price_levels)
queue_match_summary = summarize_price_level_queue_matches(price_levels)

print("Initial price-level observations:", len(price_levels))
print("Disappeared rows:", int(price_levels["disappeared"].sum()))
print("Censored rows:", int(price_levels["censored"].sum()))
print("Censored by depth window:", int(price_levels["fell_below_window"].sum()))
print("Survived to end:", int(price_levels["survived_to_end"].sum()))
print("Trade-matched rows:", int(price_levels["trade_matched"].sum()))
print("Queue-matched rows:", int(price_levels["queue_matched"].sum()))

display(price_level_summary)
display(queue_match_summary)


Initial price-level observations: 14457860
Disappeared rows: 11074252
Censored rows: 3383608
Censored by depth window: 3382031
Survived to end: 1577
Trade-matched rows: 1853016
Queue-matched rows: 10458353


,aggressor_side,level,n,disappeared_share,censored_share,censored_by_depth_share,survived_to_end_share,median_time_to_disappearance_ms,p90_time_to_disappearance_ms,mean_trade_explained_ratio,mean_residual_nontrade_ratio
0,buy,1,722893,0.853922,0.146078,0.145922,0.000156,1699.0,9700.0,0.224417,0.775583
1,buy,2,722893,0.904059,0.095941,0.095863,0.000077,1000.0,6500.0,0.229381,0.770619
2,buy,3,722893,0.854919,0.145081,0.145046,0.000035,1100.0,7400.0,0.170225,0.829775
3,buy,4,722893,0.825853,0.174147,0.174087,0.000061,1100.0,7500.0,0.161546,0.838454
4,buy,5,722893,0.789898,0.210102,0.210055,0.000047,1200.0,8300.0,0.175203,0.824797
5,buy,6,722893,0.756700,0.243300,0.243223,0.000077,1300.0,8700.0,0.179757,0.820243
6,buy,7,722893,0.723673,0.276327,0.276168,0.000159,1400.0,9200.0,0.189463,0.810537
7,buy,8,722893,0.694070,0.305930,0.305777,0.000154,1500.0,9600.0,0.201528,0.798472
8,buy,9,722893,0.670415,0.329585,0.329437,0.000148,1501.0,9900.0,0.210940,0.789060
9,buy,10,722893,0.648101,0.351899,0.351780,0.000119,1600.0,10100.0,0.213196,0.786804


,aggressor_side,level,n,trade_match_share,queue_match_share,median_time_to_trade_matched_ms,p90_time_to_trade_matched_ms,median_time_to_queue_matched_ms,p90_time_to_queue_matched_ms
0,buy,1,722893,0.093845,0.922249,1035.0,9417.1,1212.0,5301.0
1,buy,2,722893,0.184091,0.838250,1103.0,8002.0,900.0,5400.0
2,buy,3,722893,0.135113,0.780389,1432.0,10702.9,1000.0,6700.0
3,buy,4,722893,0.122511,0.756343,1643.0,10273.0,1098.0,6901.0
4,buy,5,722893,0.127499,0.719853,1927.0,12530.6,1171.0,7799.0
5,buy,6,722893,0.124501,0.695555,2226.0,13447.0,1201.0,8100.0
6,buy,7,722893,0.125045,0.669717,2493.0,13369.7,1303.0,8601.0
7,buy,8,722893,0.128572,0.647182,2810.0,14629.7,1400.0,9061.0
8,buy,9,722893,0.129227,0.628527,3085.0,15605.0,1500.0,9300.0
9,buy,10,722893,0.127799,0.609163,3255.0,15753.0,1500.0,9461.1


## How To Read The Output

A row in `price_levels` means:

- at `book_recv_seq`, a given side and initial `level` had `initial_price` and `initial_qty`
- we followed that exact price forward, even if it reranked to another level
- `status` tells whether it disappeared, was censored by depth, or survived to the end of the sample
- `censored` means the disappearance event was not observed
- `censor_reason` is either `depth_window` or `sample_end` for censored rows
- `delay_ms` is the time until the observed disappearance or censoring time
- `observed_trade_qty_at_price` is the aggressive volume seen at that exact price before termination
- `trade_matched` / `time_to_trade_matched_ms` tell you when trade flow alone first reached the initial displayed quantity
- `queue_matched` / `time_to_queue_matched_ms` tell you when trade flow plus implied non-trade removal first reached the initial displayed quantity
- `trade_explained_ratio` is the share of the initial displayed quantity that can be explained by observed trades.
- `residual_nontrade_ratio` is the remaining share for disappeared rows; interpret it as cancellation / replacement / other non-trade removal pressure.
- `trade_match_share` and `queue_match_share` are row fractions, not volume shares.
- `queue_match_share - trade_match_share` is a useful proxy for non-trade turnover, but it should not be read as a literal cancel-rate estimate.


In [4]:
columns = [
    "book_recv_seq",
    "book_side",
    "aggressor_side",
    "level",
    "initial_price",
    "initial_qty",
    "status",
    "censored",
    "censor_reason",
    "delay_ms",
    "observed_trade_qty_at_price",
    "trade_matched",
    "time_to_trade_matched_ms",
    "queue_matched",
    "time_to_queue_matched_ms",
    "trade_explained_ratio",
    "residual_nontrade_ratio",
]

display(price_levels.loc[:, columns].head(15))

survival_summary = price_level_summary.loc[
    :,
    [
        "aggressor_side",
        "level",
        "n",
        "disappeared_share",
        "censored_share",
        "censored_by_depth_share",
        "survived_to_end_share",
        "median_time_to_disappearance_ms",
        "mean_trade_explained_ratio",
        "mean_residual_nontrade_ratio",
    ],
]
display(survival_summary)

queue_summary = queue_match_summary.loc[
    :,
    [
        "aggressor_side",
        "level",
        "n",
        "trade_match_share",
        "queue_match_share",
        "median_time_to_trade_matched_ms",
        "median_time_to_queue_matched_ms",
    ],
]
display(queue_summary)


,book_recv_seq,book_side,aggressor_side,level,initial_price,initial_qty,status,censored,censor_reason,delay_ms,observed_trade_qty_at_price,trade_matched,time_to_trade_matched_ms,queue_matched,time_to_queue_matched_ms,trade_explained_ratio,residual_nontrade_ratio
0,8,bid,sell,1,66704.43,0.24012,disappeared,False,,400.0,0.00123,False,NaN,True,400.0,0.005122,0.994878
1,11,bid,sell,1,66704.43,0.24012,disappeared,False,,300.0,0.00123,False,NaN,True,300.0,0.005122,0.994878
2,12,bid,sell,1,66704.43,0.23310,disappeared,False,,200.0,0.00123,False,NaN,True,200.0,0.005277,0.994723
3,15,bid,sell,1,66704.43,0.17070,disappeared,False,,100.0,0.00123,False,NaN,True,100.0,0.007206,0.992794
4,21,bid,sell,1,66701.67,0.00057,disappeared,False,,100.0,0.00057,True,4.0,True,4.0,1.000000,0.000000
5,27,bid,sell,1,66698.38,0.17938,disappeared,False,,400.0,0.00515,False,NaN,True,400.0,0.028710,0.971290
6,28,bid,sell,1,66698.38,0.24178,disappeared,False,,299.0,0.00515,False,NaN,True,299.0,0.021300,0.978700
7,29,bid,sell,1,66698.38,0.24178,disappeared,False,,200.0,0.00515,False,NaN,True,200.0,0.021300,0.978700
8,30,bid,sell,1,66698.38,0.09709,disappeared,False,,100.0,0.00515,False,NaN,True,100.0,0.053044,0.946956
9,33,bid,sell,1,66698.01,0.07460,disappeared,False,,600.0,0.00215,False,NaN,True,400.0,0.028820,0.971180


,aggressor_side,level,n,disappeared_share,censored_share,censored_by_depth_share,survived_to_end_share,median_time_to_disappearance_ms,mean_trade_explained_ratio,mean_residual_nontrade_ratio
0,buy,1,722893,0.853922,0.146078,0.145922,0.000156,1699.0,0.224417,0.775583
1,buy,2,722893,0.904059,0.095941,0.095863,0.000077,1000.0,0.229381,0.770619
2,buy,3,722893,0.854919,0.145081,0.145046,0.000035,1100.0,0.170225,0.829775
3,buy,4,722893,0.825853,0.174147,0.174087,0.000061,1100.0,0.161546,0.838454
4,buy,5,722893,0.789898,0.210102,0.210055,0.000047,1200.0,0.175203,0.824797
5,buy,6,722893,0.756700,0.243300,0.243223,0.000077,1300.0,0.179757,0.820243
6,buy,7,722893,0.723673,0.276327,0.276168,0.000159,1400.0,0.189463,0.810537
7,buy,8,722893,0.694070,0.305930,0.305777,0.000154,1500.0,0.201528,0.798472
8,buy,9,722893,0.670415,0.329585,0.329437,0.000148,1501.0,0.210940,0.789060
9,buy,10,722893,0.648101,0.351899,0.351780,0.000119,1600.0,0.213196,0.786804


,aggressor_side,level,n,trade_match_share,queue_match_share,median_time_to_trade_matched_ms,median_time_to_queue_matched_ms
0,buy,1,722893,0.093845,0.922249,1035.0,1212.0
1,buy,2,722893,0.184091,0.838250,1103.0,900.0
2,buy,3,722893,0.135113,0.780389,1432.0,1000.0
3,buy,4,722893,0.122511,0.756343,1643.0,1098.0
4,buy,5,722893,0.127499,0.719853,1927.0,1171.0
5,buy,6,722893,0.124501,0.695555,2226.0,1201.0
6,buy,7,722893,0.125045,0.669717,2493.0,1303.0
7,buy,8,722893,0.128572,0.647182,2810.0,1400.0
8,buy,9,722893,0.129227,0.628527,3085.0,1500.0
9,buy,10,722893,0.127799,0.609163,3255.0,1500.0


## Edge Cases

These slices are useful as a sanity check on real data. They highlight rows where interpretation is easy to get wrong if we only look at aggregate summaries.


In [5]:
edge_columns = [
    "book_recv_seq",
    "book_side",
    "level",
    "initial_price",
    "initial_qty",
    "status",
    "censor_reason",
    "delay_ms",
    "observed_trade_qty_at_price",
    "trade_matched",
    "time_to_trade_matched_ms",
    "queue_matched",
    "time_to_queue_matched_ms",
    "trade_explained_ratio",
    "residual_nontrade_ratio",
]

queue_only = price_levels[price_levels["queue_matched"] & ~price_levels["trade_matched"]]
depth_censored = price_levels[price_levels["censor_reason"] == "depth_window"]
low_trade_explained = price_levels[price_levels["disappeared"] & (price_levels["trade_explained_ratio"] < 0.5)]

print("Queue matched but not trade matched:", len(queue_only))
display(queue_only.loc[:, edge_columns].head(10))

print("Censored by depth window:", len(depth_censored))
display(depth_censored.loc[:, edge_columns].head(10))

print("Disappeared with low trade-explained ratio:", len(low_trade_explained))
display(low_trade_explained.loc[:, edge_columns].head(10))


Queue matched but not trade matched: 8742989


,book_recv_seq,book_side,level,initial_price,initial_qty,status,censor_reason,delay_ms,observed_trade_qty_at_price,trade_matched,time_to_trade_matched_ms,queue_matched,time_to_queue_matched_ms,trade_explained_ratio,residual_nontrade_ratio
0,8,bid,1,66704.43,0.24012,disappeared,,400.0,0.00123,False,NaN,True,400.0,0.005122,0.994878
1,11,bid,1,66704.43,0.24012,disappeared,,300.0,0.00123,False,NaN,True,300.0,0.005122,0.994878
2,12,bid,1,66704.43,0.23310,disappeared,,200.0,0.00123,False,NaN,True,200.0,0.005277,0.994723
3,15,bid,1,66704.43,0.17070,disappeared,,100.0,0.00123,False,NaN,True,100.0,0.007206,0.992794
5,27,bid,1,66698.38,0.17938,disappeared,,400.0,0.00515,False,NaN,True,400.0,0.028710,0.971290
6,28,bid,1,66698.38,0.24178,disappeared,,299.0,0.00515,False,NaN,True,299.0,0.021300,0.978700
7,29,bid,1,66698.38,0.24178,disappeared,,200.0,0.00515,False,NaN,True,200.0,0.021300,0.978700
8,30,bid,1,66698.38,0.09709,disappeared,,100.0,0.00515,False,NaN,True,100.0,0.053044,0.946956
9,33,bid,1,66698.01,0.07460,disappeared,,600.0,0.00215,False,NaN,True,400.0,0.028820,0.971180
10,34,bid,1,66698.01,0.07476,disappeared,,500.0,0.00215,False,NaN,True,300.0,0.028759,0.971241


Censored by depth window: 3382031


,book_recv_seq,book_side,level,initial_price,initial_qty,status,censor_reason,delay_ms,observed_trade_qty_at_price,trade_matched,time_to_trade_matched_ms,queue_matched,time_to_queue_matched_ms,trade_explained_ratio,residual_nontrade_ratio
173,1046,bid,1,66603.74,0.25065,fell_below_window,depth_window,401.0,0.00009,False,NaN,False,NaN,0.000359,NaN
174,1049,bid,1,66603.74,0.42447,fell_below_window,depth_window,301.0,0.00000,False,NaN,False,NaN,0.000000,NaN
175,1050,bid,1,66604.96,0.36249,fell_below_window,depth_window,300.0,0.00000,False,NaN,False,NaN,0.000000,NaN
176,1051,bid,1,66607.99,0.21000,fell_below_window,depth_window,199.0,0.00000,False,NaN,False,NaN,0.000000,NaN
374,1814,bid,1,66555.56,0.25365,fell_below_window,depth_window,9101.0,0.01186,False,NaN,True,3101.0,0.046757,NaN
394,1850,bid,1,66555.56,0.02362,fell_below_window,depth_window,7101.0,0.01186,False,NaN,True,1101.0,0.502117,NaN
395,1851,bid,1,66555.56,0.02362,fell_below_window,depth_window,7001.0,0.01186,False,NaN,True,1001.0,0.502117,NaN
396,1852,bid,1,66555.56,0.02362,fell_below_window,depth_window,6901.0,0.01186,False,NaN,True,901.0,0.502117,NaN
397,1854,bid,1,66555.56,0.11195,fell_below_window,depth_window,6801.0,0.00294,False,NaN,True,801.0,0.026262,NaN
398,1857,bid,1,66555.56,0.21628,fell_below_window,depth_window,6701.0,0.00294,False,NaN,True,1499.0,0.013593,NaN


Disappeared with low trade-explained ratio: 8973528


,book_recv_seq,book_side,level,initial_price,initial_qty,status,censor_reason,delay_ms,observed_trade_qty_at_price,trade_matched,time_to_trade_matched_ms,queue_matched,time_to_queue_matched_ms,trade_explained_ratio,residual_nontrade_ratio
0,8,bid,1,66704.43,0.24012,disappeared,,400.0,0.00123,False,NaN,True,400.0,0.005122,0.994878
1,11,bid,1,66704.43,0.24012,disappeared,,300.0,0.00123,False,NaN,True,300.0,0.005122,0.994878
2,12,bid,1,66704.43,0.23310,disappeared,,200.0,0.00123,False,NaN,True,200.0,0.005277,0.994723
3,15,bid,1,66704.43,0.17070,disappeared,,100.0,0.00123,False,NaN,True,100.0,0.007206,0.992794
5,27,bid,1,66698.38,0.17938,disappeared,,400.0,0.00515,False,NaN,True,400.0,0.028710,0.971290
6,28,bid,1,66698.38,0.24178,disappeared,,299.0,0.00515,False,NaN,True,299.0,0.021300,0.978700
7,29,bid,1,66698.38,0.24178,disappeared,,200.0,0.00515,False,NaN,True,200.0,0.021300,0.978700
8,30,bid,1,66698.38,0.09709,disappeared,,100.0,0.00515,False,NaN,True,100.0,0.053044,0.946956
9,33,bid,1,66698.01,0.07460,disappeared,,600.0,0.00215,False,NaN,True,400.0,0.028820,0.971180
10,34,bid,1,66698.01,0.07476,disappeared,,500.0,0.00215,False,NaN,True,300.0,0.028759,0.971241
